In [4]:
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.hamiltonians import FermiHubbardModel
from qiskit_nature.second_q.hamiltonians.lattices import HexagonalLattice

epsilon = 0.0
gamma_0 = 1.0
U_0 = 0.0

num_sites = 6
num_spin_orbitals = 2*num_sites

lattice = HexagonalLattice(1, 1, edge_parameter=gamma_0, onsite_parameter=epsilon)
model = FermiHubbardModel(lattice, onsite_interaction=U_0)
fermionic_hamiltonian = model.second_q_op()
jw = JordanWignerMapper()
qubit_hamiltonian = jw.map(fermionic_hamiltonian)

In [7]:
from qiskit.synthesis import SuzukiTrotter
from qiskit.circuit.library import PauliEvolutionGate
from qiskit_nature.second_q.circuit.library import HartreeFock

t = 0.2
N_trot = 15
st = SuzukiTrotter(reps=N_trot)
evolution = PauliEvolutionGate(qubit_hamiltonian, time=t)
evolution_circuit = st.synthesize(evolution)
hf_circuit = HartreeFock(num_spatial_orbitals=num_sites, num_particles=(5, 0), qubit_mapper=jw)

In [8]:
from qiskit_algorithms import IterativePhaseEstimation
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import Sampler
from qiskit import transpile
import numpy as np

backend, sampler = AerSimulator(), Sampler()
evolution_circuit, hf_circuit = transpile(evolution_circuit, backend=backend), transpile(hf_circuit, backend=backend)
iqpe = IterativePhaseEstimation(num_iterations=5, sampler=sampler)
result = iqpe.estimate(unitary=evolution_circuit, state_preparation=hf_circuit)

energy = -2*np.pi * result.phase / t
print(energy)

-1.9634954084936207
